# Deposit Study 2027v101 MV Workflow Graph

An interactive, layered map of the model validation (MV) package that `studies/mv_report` builds for study **2027v101** (parent 2027v16, snapshot 2026-06-30). It follows the version and production sources and the `inputs/mv/mv_config.xlsx` tabs through the 8 MV stages to the `finance.sandbox.mv_*_2027v101` tables, the `outputs/mv/` workbooks, the workflow document and the MV dashboard. Click a node to see its logic, parameters and lineage. The **Run order** tab walks the stages in sequence; the **Issues** tab lists the open findings.

The last mv_report run was in **preview mode** (`preview_with_current_product = Y`): no `new_product` is assigned yet, so stages 5 to 8 show current products.

Same style as `Deposit Study 2027v16 Lineage Graph`; the render cell reuses that notebook's HTML template, so keep both notebooks in `liquidity/resources/`. This notebook only documents the MV report and sits outside the version folder.

In [0]:
# Deposit Study 2027v101 MV workflow graph definition.
# Built by reading studies/mv_report, the 2027v101 MV outputs and the mv_*_2027v101 sandbox tables (2026-10-25).
# Edit this cell to add/rename nodes; the render cell below picks up changes automatically.

LAYERS = [
    ("src",    "Version & production sources"),
    ("inputs", "Inputs & mv_config.xlsx"),
    ("fw",     "Framework & helpers"),
    ("s1",     "1. Accounts"),
    ("s2",     "2. Account metrics"),
    ("s3",     "3. Subproduct betas"),
    ("s4",     "4. Product selection"),
    ("s5",     "5. Product maps"),
    ("s6",     "6. Product betas"),
    ("s7",     "7. Beta review"),
    ("s8",     "8. Final betas"),
    ("docs",   "Docs & dashboard"),
]
L = {k: i for i, (k, _) in enumerate(LAYERS)}

NODES = []
def node(id, label, layer, kind, entity, desc, logic=None, params=None, warn=None, src=None, fq=None):
    NODES.append(dict(id=id, label=label, layer=L[layer], kind=kind, entity=entity, desc=desc,
                      logic=logic or [], params=params or {}, warn=warn, src=src, fq=fq or label))

TAG = "2027v101"
SB = "finance.sandbox."
V = "studies/2027/2027v101/"
OUT = V + "outputs/mv/"
NB = "studies/mv_report"

# ---------------------------------------------------------------- L0 sources
node("src_lpnfp", "lpnfp_nmds", "src", "table", "PNFP",
     "The version's PNFP account table built by notebook 01. Stage 1 takes the snapshot rows, excluding internal and closed accounts as lpnfp_nmds_agg does.",
     fq=SB + "lpnfp_nmds_" + TAG,
     warn="130 PNFP account rows ($4.85M) appear twice because dim_nmd has two rows for CP_000_00. MV de-duplicates them; nmds_agg does not.")
node("src_lsnv", "lsnv_nmds", "src", "table", "SNV",
     "Production SNV account-level deposits, read directly ({{src_lsnv_nmds}}).", fq="finance.pricing_analytics.lsnv_nmds")
node("src_dim", "dim_nmd", "src", "table", "Both",
     "The version's product dimension: subproductid + legacy_bank + pf / bdsp flags -> product_group, product, product_num.",
     fq=SB + "dim_nmd_" + TAG,
     warn="Duplicate key: PNFP CP_000_00 (IB DDA Commercial Standard) has 2 rows. Accounts with no dim_nmd row become UNMAPPED.")
node("src_agg", "nmds_agg", "src", "table", "Both",
     "The version's monthly aggregate by bank / subproduct / product (eopbook, avgbook, sumprod_eoprate). It is the history behind every beta, the same table notebook 03 uses, and the stage 1 reconciliation baseline.",
     fq=SB + "nmds_agg_" + TAG)
node("src_ffr", "fed_funds_matrix", "src", "table", "Both",
     "Daily fed funds (UP_FFR, EFFR). betas.load_effr_monthly turns it into month-end values.",
     fq="finance.operational_deposits.fed_funds_matrix")
node("src_ccalc", "ccalc_beta_results", "src", "table", "Both",
     "Notebook 03's beta results for the version. Used only by check 6b to prove the MV engine reproduces 03.",
     fq=SB + "ccalc_beta_results_" + TAG)
node("src_nb", "notebooks 01 + 02", "src", "notebook", "Both",
     "The version's build and mapping notebooks. Stage 1b exports every sql(...) / sqlq(...) block and renders the tokens for 2027v101.",
     fq=V + "notebooks/01_build_tables, 02_product_mapping")

# ---------------------------------------------------------------- L1 inputs
node("in_widget", "version widget", "inputs", "config", "Both",
     "Picks the study version. mv_report finds _shared/deposit_utils next to itself, lists the versions and loads the one chosen.",
     params={"version": TAG, "parent": "2027v16"})
node("in_params_json", "study_params.json", "inputs", "config", "Both",
     "Version dates and schemas.", params={"snapshot_date": "2026-06-30"}, fq=V + "inputs/config/study_params.json")
node("in_cycles_csv", "rate_cycles.csv", "inputs", "config", "Both",
     "Named rate cycles (start, end, short code). Feeds the TTC window and the up / down cycles.", fq=V + "inputs/config/rate_cycles.csv")
node("cfg_params", "mv_config: params", "inputs", "config", "Both",
     "Method settings. Edit the value column only; re-run mv_report from the top after a change.",
     params={"ttc_cycle": "TTC PD (2019-01 to 2026-07, 91 months)", "lags": "0,1,2,3", "poly_rate_col": "UP_FFR",
             "avg_r2_tolerance": "5", "updown_lag": "0", "min_months_ttc": "24", "min_balance": "1,000,000",
             "snv_bdsp_accounts": "16501007815721, 16501007815713", "snv_excluded_subproducts": "050_082_00, 050_000_00",
             "preview_with_current_product": "Y (last run)"},
     fq=V + "inputs/mv/mv_config.xlsx [params]",
     warn="preview_with_current_product = Y: every subproduct uses its current product, so stages 5-8 do not show the new product structure yet.")
node("cfg_cycles", "mv_config: cycles", "inputs", "config", "Both",
     "One row per rate_cycles.csv cycle. direction (UP / DOWN / TTC) is defaulted from the sign of the UP_FFR change; include = Y / N picks the cycles averaged into avg up / avg down.",
     params={"UP": "500bpsUP", "DOWN": "125bpsdown, MRT_DOWN"}, fq=V + "inputs/mv/mv_config.xlsx [cycles]")
node("cfg_assign", "mv_config: product_assignment", "inputs", "config", "Both",
     "YOUR INPUT. One row per sub_key (bank | subproductid | current product), refreshed by stage 4. Type the new product in new_product; rows with the same text (trimmed, case-sensitive) become one product, and a product may combine PNFP and SNV.",
     params={"rows": "540", "assigned": "0", "you edit": "new_product, notes"}, fq=V + "inputs/mv/mv_config.xlsx [product_assignment]",
     warn="0 of 540 rows assigned ($85.50B unassigned).")
node("cfg_names", "mv_config: product_names", "inputs", "config", "Both",
     "YOUR INPUT. Every distinct new_product. Fill consolidated_name to merge or rename products without retyping assignments.",
     params={"you edit": "consolidated_name, notes"}, fq=V + "inputs/mv/mv_config.xlsx [product_names]")
node("cfg_over", "mv_config: final_overrides", "inputs", "config", "Both",
     "YOUR INPUT. One row per product, seeded by stage 7. override_method = LM, POLY, AVG (optional override_lag) or MANUAL with manual_beta; reason required.",
     params={"you edit": "override_method, override_lag, manual_beta, reason"}, fq=V + "inputs/mv/mv_config.xlsx [final_overrides]")

# ---------------------------------------------------------------- L2 framework
node("fw_study", "study.Study (S)", "fw", "function", "Both",
     "Version object: S.tag, S.snapshot_date, S.table(name), S.input / S.output paths, {{tokens}} for sql() / sqlq().",
     params={"MV_TABLES": "10 mv_* tables, kept out of SANDBOX_TABLES so clones do not copy MV results"}, fq="_shared/deposit_utils/study.py")
node("fw_effr", "betas.load_effr_monthly", "fw", "function", "Both",
     "Month-end UP_FFR / EFFR series, with build_named_cycles giving the month list for each cycle.", fq="_shared/deposit_utils/betas.py")
node("fw_betas", "betas: LM / poly / endpoint", "fw", "function", "Both",
     "The notebook 03 engine: linear Lm_Coef + R2, cubic Poly_Beta + R2, endpoint_beta_lag (change in WAR / change in fed funds).",
     fq="_shared/deposit_utils/betas.py",
     warn="endpoint_beta_lag returns NaN when the lagged start date is a non-month-end last observation, e.g. 125bpsdown and 500bpsUP at lag 1 and MRT_DOWN at lag 2. Notebook 03 is affected too.")
node("fw_grid", "beta_grid / war_pivot", "fw", "function", "Both",
     "mv_report helpers. war_pivot: long (key, eopdate, war) -> one row per key, one column per month (same layout as 03). beta_grid: LM and poly at every lag on ttc_cycle, endpoint betas for each UP / DOWN cycle, avg_up_<lag> / avg_down_<lag>.",
     fq=NB + " cell 3")
node("fw_rec", "recommend()", "fw", "function", "Both",
     "The recommendation rule.",
     logic=["best LM lag = highest lm_r2; best poly lag = highest poly_r2",
            "if |best LM R2 - best poly R2| <= avg_r2_tolerance: AVG(best LM, best poly)",
            "else the method with the higher R2",
            "NONE when neither has an R2"], fq=NB + " cell 3")
node("fw_io", "read_cfg / write_cfg / save_table", "fw", "function", "Both",
     "read_cfg reads the workbook (keeps extra columns); write_cfg rewrites every tab plus README; write_xlsx sizes columns and freezes headers; save_table overwrites finance.sandbox.<name>_<tag> (only for MV_TABLES).",
     fq=NB + " config cell")

# ---------------------------------------------------------------- L3 stage 1
node("s1_sql", "1a MV_ACCOUNTS_SQL", "s1", "query", "Both",
     "One row per account at snapshot_date, classified exactly as notebook 01.",
     logic=["PNFP: SELECT DISTINCT from lpnfp_nmds at the snapshot, internal and closed accounts excluded",
            "SNV: lsnv_nmds joined to dim_nmd on subproduct + BDSP flag (BDSP accounts from params); excluded subproducts removed",
            "sub_key = legacy_bank | subproductid | COALESCE(product, 'UNMAPPED'); mapped = dim_nmd match"])
node("t_accounts", "mv_accounts", "s1", "table", "Both",
     "Stage 1 table: 1,124,941 accounts, $85,500,947,222 (PNFP 426,248 / $47.25B, SNV 698,693 / $38.26B). 23 columns.",
     fq=SB + "mv_accounts_" + TAG)
node("s1_recon", "1a recon vs nmds_agg", "s1", "query", "Both",
     "Accounts and balance by bank vs nmds_agg at the snapshot; unmapped balance; duplicate dim_nmd keys.",
     params={"PNFP diff": "-$4,848,952 (130 duplicated rows)", "SNV diff": "$0", "unmapped": "32 PNFP accounts, $15,163,602"},
     warn="32 PNFP accounts ($15.16M, 3D_000_00 / 4F_000_00, prime-787 / override-0) have no dim_nmd row and are UNMAPPED.")
node("s1_sqlmd", "account_mapping_sql .md/.sql", "s1", "file", "Both",
     "11 rendered SQL blocks from notebooks 01 and 02 plus the MV extract.", fq=OUT + "01_accounts/account_mapping_sql_" + TAG + ".md")
node("s1_dict", "account_column_dictionary.csv", "s1", "file", "Both",
     "23 columns: PNFP source, SNV source, rule, type, null count, distinct count, sample value.", fq=OUT + "01_accounts/account_column_dictionary_" + TAG + ".csv")
node("s1_xlsx", "accounts.xlsx", "s1", "file", "Both",
     "Account extract summary by bank / subproduct / product (310 rows).", fq=OUT + "01_accounts/accounts_" + TAG + ".xlsx")

# ---------------------------------------------------------------- L4 stage 2
node("t_war", "mv_account_war", "s2", "table", "Both",
     "Monthly balance and WAR per sub_key over the full history (29,220 rows, 104 months). war = SUM(sumprod_eoprate) / SUM(eopbook).",
     fq=SB + "mv_account_war_" + TAG)
node("t_metrics", "mv_account_metrics", "s2", "table", "Both",
     "Snapshot metrics per sub_key (306 rows, $85.50B, ties to stage 1): accounts, eopbook, avgbook, WAR, rate min / p10-p90 / max / std, zero-rate balance share, share of product balance, months of history.",
     fq=SB + "mv_account_metrics_" + TAG)
node("s2_files", "account_metrics.xlsx + war csv", "s2", "file", "Both",
     "account_metrics_2027v101.xlsx and account_war_history_2027v101.csv.", fq=OUT + "02_account_metrics/")

# ---------------------------------------------------------------- L5 stage 3
node("s3_run", "3 beta grid per sub_key", "s3", "function", "Both",
     "war_pivot(mv_account_war) -> beta_grid -> recommend(), then the flags.",
     params={"flag_thin_history": "months_ttc < 24 (214 sub_keys)", "flag_low_balance": "balance < $1M (302 sub_keys)"})
node("t_abetas", "mv_account_betas", "s3", "table", "Both",
     "540 sub_keys (every one with history, including those with no balance at the snapshot), 61 columns: lm / poly / R2 at lags 0-3, endpoint betas per cycle and lag, avg up / down, best lags, recommendation, flags.",
     fq=SB + "mv_account_betas_" + TAG,
     warn="15 sub_keys get recommended_method NONE (all $0 at the snapshot). 16 have a negative best R2 (poly R2 as low as -52.8M) from near-flat WAR series.")
node("s3_xlsx", "account_betas.xlsx", "s3", "file", "Both", "Subproduct beta workbook.", fq=OUT + "03_account_betas/account_betas_" + TAG + ".xlsx")

# ---------------------------------------------------------------- L6 stage 4
node("s4_refresh", "4 refresh workbook", "s4", "function", "Both",
     "Rewrites product_assignment from mv_account_betas, keeping your new_product / notes by sub_key; rebuilds product_names; reports assignments whose sub_key disappeared.",
     logic=["base = mv_account_betas context columns", "merge old new_product / notes on sub_key",
            "product_names = distinct new_product (+ kept consolidated_name)", "write_cfg"])
node("s4_select", "mv_product resolution", "s4", "function", "Both",
     "mv_product = consolidated_name, else new_product, else current product (preview only), else unassigned. assignment_source records which applied.",
     params={"assignment_source": "ASSIGNED / PREVIEW_CURRENT_PRODUCT / UNASSIGNED"})
node("t_sel", "mv_product_selection", "s4", "table", "Both",
     "540 rows: sub_key, bank, subproductid, current_product, product_group, snapshot_balance, new_product, notes, assignment_source, mv_product. All 540 are PREVIEW_CURRENT_PRODUCT.",
     fq=SB + "mv_product_selection_" + TAG)
node("s4_gate", "READY gate", "s4", "query", "Both",
     "Stages 5-8 run only if at least one sub_key has an mv_product (an assignment, or preview mode). Otherwise the notebook exits after stage 4.")
node("s4_xlsx", "product_selection.xlsx", "s4", "file", "Both", "Assignment snapshot.", fq=OUT + "04_product_selection/product_selection_" + TAG + ".xlsx")

# ---------------------------------------------------------------- L7 stage 5
node("t_map", "mv_product_map", "s5", "table", "Both",
     "Every snapshot account with new_product, mv_product and assignment_source (1,124,941 rows).", fq=SB + "mv_product_map_" + TAG)
node("s5_sum", "product summary", "s5", "view", "Both",
     "Per mv_product: banks, subproducts, accounts, balance, WAR and the current products feeding it. 37 products hold snapshot balance; the largest is NIB DDA Commercial at $14.43B.")
node("s5_recon", "5 recon assigned + unassigned", "s5", "query", "Both",
     "Assigned + unassigned = all accounts. v101: ties at 1,124,941 accounts and $85,500,947,222; unassigned $0 (preview).")
node("s5_files", "product_map.xlsx + account csv", "s5", "file", "Both",
     "product_map_2027v101.xlsx and account_product_map_2027v101.csv.", fq=OUT + "05_product_maps/")

# ---------------------------------------------------------------- L8 stage 6
node("t_pwar", "mv_product_war", "s6", "table", "Both",
     "Monthly WAR per mv_product = balance-weighted WAR of its assigned sub_keys, with today's assignment applied to every month (3,622 rows, 38 products).",
     fq=SB + "mv_product_war_" + TAG)
node("t_pbetas", "mv_product_betas", "s6", "table", "Both",
     "38 products, 46 columns: makeup, snapshot balance and WAR, lm / poly / R2 at lags 0-3, endpoint betas per cycle and lag, avg up / down, months_ttc.",
     fq=SB + "mv_product_betas_" + TAG)
node("s6_check", "6b check vs notebook 03", "s6", "query", "Both",
     "When a product has exactly the same sub_keys as a current product (always true in preview), its betas must equal ccalc_beta_results.",
     params={"compared": "37 products, 20 columns", "result": "MATCH (max diff 0, no NULL mismatches)"})
node("s6_xlsx", "product_beta_report.xlsx", "s6", "file", "Both",
     "Tabs: Product betas, Up and down by cycle, Monthly WAR, Settings, Cycles.", fq=OUT + "06_product_betas/product_beta_report_" + TAG + ".xlsx")

# ---------------------------------------------------------------- L9 stage 7
node("t_review", "mv_beta_review", "s7", "table", "Both",
     "38 rows: every candidate, best LM / poly, their average, recommended method / beta / R2 / reason, and flags (thin history, low balance, preview).",
     params={"AVG": "21 products, $49.61B", "POLY_LAG3": "6, $20.30B", "POLY_LAG0": "9, $15.45B", "LM_LAG0": "2, $0.14B"},
     fq=SB + "mv_beta_review_" + TAG)
node("s7_tabs", "beta_review.xlsx", "s7", "file", "Both",
     "All products tab plus one tab per product listing every candidate, with the recommended one marked.", fq=OUT + "07_beta_review/beta_review_" + TAG + ".xlsx")
node("s7_seed", "seed final_overrides", "s7", "function", "Both",
     "Writes one row per product (recommended method and beta) into final_overrides, keeping existing overrides.")

# ---------------------------------------------------------------- L10 stage 8
node("s8_apply", "8 apply overrides", "s8", "function", "Both",
     "The final method per product.",
     logic=["blank override_method -> recommended (final_source RULE)",
            "LM / POLY -> that method at override_lag, else its best-R2 lag",
            "AVG -> mean of LM and POLY at the chosen lags",
            "MANUAL -> manual_beta, R2 blank",
            "any other value stops the run; an override without a reason is flagged"])
node("t_final", "mv_final_betas", "s8", "table", "Both",
     "38 rows: recommended and final method / beta / R2, avg up / down, final_source, override reason, flags, study, snapshot_date, ttc_cycle. 0 overrides so far.",
     fq=SB + "mv_final_betas_" + TAG)
node("s8_csv", "final_betas.csv", "s8", "file", "Both", "Flat export of mv_final_betas.", fq=OUT + "08_final_betas/final_betas_" + TAG + ".csv")
node("s8_sum", "mv_summary.xlsx", "s8", "file", "Both",
     "The MV package: Summary, Final betas, Beta review, Product betas, Product map, Subproduct betas, Reconciliation, Settings, Cycles.",
     fq=OUT + "08_final_betas/mv_summary_" + TAG + ".xlsx")

# ---------------------------------------------------------------- L11 docs
node("d_md", "mv_workflow_2027v101.md", "docs", "file", "Both",
     "Stage-by-stage write-up: inputs, logic, tables, files, checks, v101 results, re-run guide, open items.", fq=OUT + "docs/mv_workflow_" + TAG + ".md")
node("d_dash", "MV dashboard", "docs", "external", "Both",
     "Deposit Study MV Report 2027v101: overview plus one page per stage, reading the mv_*_2027v101 tables.")

# ---------------------------------------------------------------- edges
EDGES = []
def e(s, t, type="d", label=None):
    EDGES.append(dict(s=s, t=t, type=type, label=label))

# config / framework
for t in ["fw_study"]: e("in_widget", t, "p", "version")
e("in_params_json", "fw_study", "p", "snapshot_date")
e("in_cycles_csv", "fw_effr", "p", "cycles")
e("in_cycles_csv", "cfg_cycles", "d", "seed rows")
e("src_ffr", "fw_effr")
e("fw_effr", "fw_betas")
e("fw_betas", "fw_grid", "o")
e("cfg_params", "fw_grid", "p", "ttc_cycle, lags, poly col")
e("cfg_cycles", "fw_grid", "p", "UP / DOWN")
e("cfg_params", "fw_rec", "p", "avg_r2_tolerance")
e("fw_io", "cfg_params", "o", "create / extend")
# stage 1
for s in ["src_lpnfp", "src_lsnv", "src_dim"]: e(s, "s1_sql")
e("cfg_params", "s1_sql", "p", "BDSP / excluded")
e("fw_study", "s1_sql", "p", "tokens")
e("s1_sql", "t_accounts")
e("t_accounts", "s1_recon", "r")
e("src_agg", "s1_recon", "r", "baseline")
e("src_dim", "s1_recon", "r", "dup keys")
e("src_nb", "s1_sqlmd", "d", "sql blocks")
e("s1_sql", "s1_sqlmd")
e("t_accounts", "s1_dict")
e("t_accounts", "s1_xlsx")
# stage 2
e("src_agg", "t_war")
e("t_accounts", "t_metrics")
e("t_war", "t_metrics", "d", "history")
e("t_metrics", "s2_files"); e("t_war", "s2_files")
# stage 3
e("t_war", "s3_run")
e("fw_grid", "s3_run", "o")
e("fw_rec", "s3_run", "o")
e("cfg_params", "s3_run", "p", "flag thresholds")
e("t_metrics", "s3_run", "d", "snapshot cols")
e("s3_run", "t_abetas")
e("t_abetas", "s3_xlsx")
# stage 4
e("t_abetas", "s4_refresh")
e("cfg_assign", "s4_refresh", "p", "new_product, notes")
e("s4_refresh", "cfg_assign", "o", "write back")
e("s4_refresh", "cfg_names", "o", "write back")
e("fw_io", "s4_refresh", "o")
e("s4_refresh", "s4_select")
e("cfg_names", "s4_select", "p", "consolidated_name")
e("cfg_params", "s4_select", "p", "preview flag")
e("s4_select", "t_sel")
e("t_sel", "s4_gate", "r")
e("t_sel", "s4_xlsx")
# stage 5
e("s4_gate", "t_map", "o", "READY")
e("t_accounts", "t_map")
e("t_sel", "t_map")
e("t_map", "s5_sum")
e("t_map", "s5_recon", "r")
e("t_accounts", "s5_recon", "r")
e("s5_sum", "s5_files"); e("t_map", "s5_files")
# stage 6
e("t_war", "t_pwar")
e("t_sel", "t_pwar", "d", "sub_key -> mv_product")
e("t_pwar", "t_pbetas")
e("fw_grid", "t_pbetas", "o", "beta_grid")
e("s5_sum", "t_pbetas", "d", "makeup")
e("t_pbetas", "s6_check", "r")
e("src_ccalc", "s6_check", "r", "03 betas")
e("t_pbetas", "s6_xlsx"); e("t_pwar", "s6_xlsx")
# stage 7
e("t_pbetas", "t_review")
e("fw_rec", "t_review", "o", "recommend")
e("cfg_params", "t_review", "p", "flag thresholds")
e("t_review", "s7_tabs")
e("t_review", "s7_seed")
e("s7_seed", "cfg_over", "o", "write back")
# stage 8
e("t_review", "s8_apply")
e("cfg_over", "s8_apply", "p", "overrides")
e("s8_apply", "t_final")
e("t_final", "s8_csv")
for s in ["t_final", "t_review", "t_pbetas", "s5_sum", "t_abetas", "s1_recon", "cfg_params", "cfg_cycles"]:
    e(s, "s8_sum", "d" if s.startswith("t_") or s.startswith("s") else "p")
# docs
for s in ["s8_sum", "s1_recon", "s6_check"]: e(s, "d_md")
for s in ["t_accounts", "t_metrics", "t_war", "t_abetas", "t_sel", "t_map", "t_pwar", "t_pbetas", "t_review", "t_final", "src_agg"]:
    e(s, "d_dash", "d")

# ---------------------------------------------------------------- run order
STAGES = [
    ("Pick version and load config", "version widget -> S; creates or extends mv_config.xlsx (params, cycles) and parses the settings.",
     ["in_widget", "in_params_json", "in_cycles_csv", "fw_study", "fw_io", "cfg_params", "cfg_cycles", "fw_effr"]),
    ("1. Accounts", "One row per account at the snapshot, reconciled to nmds_agg; mapping SQL, column dictionary, extract.",
     ["src_lpnfp", "src_lsnv", "src_dim", "src_agg", "src_nb", "s1_sql", "t_accounts", "s1_recon", "s1_sqlmd", "s1_dict", "s1_xlsx"]),
    ("2. Account metrics", "Monthly WAR history and snapshot metrics per sub_key.",
     ["src_agg", "t_war", "t_metrics", "s2_files"]),
    ("3. Subproduct betas", "notebook 03 engine on each sub_key: LM / poly lags 0-3, endpoint up / down, recommendation, flags.",
     ["t_war", "fw_betas", "fw_grid", "fw_rec", "s3_run", "t_abetas", "s3_xlsx"]),
    ("4. Product selection (your input)", "Refresh product_assignment / product_names, resolve mv_product, READY gate.",
     ["t_abetas", "s4_refresh", "cfg_assign", "cfg_names", "s4_select", "t_sel", "s4_gate", "s4_xlsx"]),
    ("5. Product maps", "Accounts -> mv_product, product summary, assigned + unassigned tie-out.",
     ["t_sel", "t_map", "s5_sum", "s5_recon", "s5_files"]),
    ("6. Product betas", "Balance-weighted product WAR, beta grid, check 6b against notebook 03.",
     ["t_pwar", "t_pbetas", "s6_check", "src_ccalc", "s6_xlsx"]),
    ("7. Beta review", "Candidates, recommended method and flags per product; seed final_overrides.",
     ["t_review", "s7_tabs", "s7_seed", "cfg_over"]),
    ("8. Final betas (your input)", "Apply overrides; write mv_final_betas, final_betas.csv and mv_summary.xlsx.",
     ["cfg_over", "s8_apply", "t_final", "s8_csv", "s8_sum"]),
    ("Docs and dashboard", "Workflow document and the MV dashboard over the mv_* tables.",
     ["d_md", "d_dash"]),
]

# ---------------------------------------------------------------- sanity checks
ids = [n["id"] for n in NODES]
assert len(ids) == len(set(ids)), "duplicate node ids"
bad = [(x["s"], x["t"]) for x in EDGES if x["s"] not in ids or x["t"] not in ids]
assert not bad, f"edges to unknown nodes: {bad}"
bad = [n for _, _, ns in STAGES for n in ns if n not in ids]
assert not bad, f"stages reference unknown nodes: {bad}"
from collections import Counter
print(f"{len(NODES)} nodes | {len(EDGES)} edges | {len(STAGES)} run-order stages | {sum(1 for n in NODES if n['warn'])} flagged issues")
for i, (k, nm) in enumerate(LAYERS):
    print(f"  L{i:<3}{nm:<34}{sum(1 for n in NODES if n['layer'] == i):>3} nodes")
print("edge types:", dict(Counter(x["type"] for x in EDGES)))

In [0]:
# Reuses the HTML / JS template from the 2027v16 lineage graph notebook (same folder), so both graphs look and behave the same.
import json, base64, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat

TEMPLATE_NB = "/Users/joshuawaller@synovus.com/liquidity/resources/Deposit Study 2027v16 Lineage Graph"
src = base64.b64decode(WorkspaceClient().workspace.export(TEMPLATE_NB, format=ExportFormat.SOURCE).content).decode("utf-8")
start = src.index('HTML = r"""') + len('HTML = r"""')
TEMPLATE = src[start:src.index('"""', start)]

# retitle for the MV workflow
for old, new in [
    ("deposit study 2027v16<span>- deposit beta study: lineage, notebooks &amp; model logic</span>",
     "deposit study 2027v101 MV workflow<span>- mv_report stages, tables, outputs &amp; checks</span>"),
    ("production sources</span>", "sources</span>"),
    ("Findings from the 2027v16 review", "Open findings (mv_report run 2026-10-25, preview mode)"),
    ("Run order - click a stage to spotlight its nodes", "MV stages - click one to spotlight its nodes"),
    ("e.g. nmds_agg, lpnfp_betas, product_config", "e.g. mv_accounts, product_assignment, recommend"),
    ("Python function / step", "Function / mv_report step"),
    ("Study notebook", "Version notebook"),
    ("Config file (inputs/)", "Config / workbook tab (your input)"),
    ("Promotion target", "Dashboard"),
]:
    assert old in TEMPLATE, f"template text not found: {old}"
    TEMPLATE = TEMPLATE.replace(old, new)

LAYER_COLORS = ["#5c6c8f", "#7a5cc8", "#2f7fd6", "#b8811a", "#128f7c", "#3a9fbf",
                "#c8532f", "#d0457a", "#6b8e23", "#8a6d3b", "#4f6d7a", "#a0527a"]
payload = {
    "layers": [{"id": k, "name": nm, "color": LAYER_COLORS[i % len(LAYER_COLORS)]} for i, (k, nm) in enumerate(LAYERS)],
    "nodes": NODES,
    "edges": EDGES,
    "stages": [{"title": t, "desc": d, "nodes": ns} for t, d, ns in STAGES],
}
data_json = json.dumps(payload).replace("</", "<\\/")
displayHTML(TEMPLATE.replace("__DATA__", data_json))